# On the Lookout with MeerQat

This notebook shows the basic MeerQat workflow.

## Where the LLM Helps

- **Deterministic validation** decides the real pass, warn, or fail result.
- **LLM review** runs afterward and adds interpretation plus suspected hidden issues.

MeerQat prefers **Instructor** for its core review path and falls back to
**LangChain** if needed.


## Configure the Local Model

MeerQat uses the LLM on every run. For the basic walkthrough, the defaults are
enough.


In [8]:
import pathlib
from tempfile import TemporaryDirectory

from meerqat import ValidationConfig, validate_dataset, write_reports

## Validate a Known-Good Dataset

The repository includes small sample datasets under `tests/data`. We can use
the valid fixture first. Because the matching `metadata.csv` lives next to the
dataset folder, MeerQat will discover it automatically.


In [9]:
data_root = pathlib.Path("../..") / "tests" / "data" / "valid_minimal"
report = validate_dataset(
    data_root / "dataset",
)

report.summary.status

'pass'

The expected result is `"pass"`.


In [10]:
report.summary.to_dict()

{'status': 'pass',
 'dataset_id': 'dataset',
 'plate_count': 1,
 'metadata_record_count': 0,
 'image_count': 1,
 'issue_counts': {},
 'ready_for_pipeline': True}

## Inspect a Broken Dataset

Now switch to a sample dataset where the XML plate identifier does not match
the folder name.


In [11]:
broken_root = pathlib.Path("../..") / "tests" / "data" / "xml_mismatch"
broken_report = validate_dataset(
    broken_root / "dataset",
)

broken_report.summary.status

'fail'

The expected result is `"fail"`. That fail status comes from deterministic
rules, not from the LLM.


After the issue list is known, the LLM can help explain it, for example by
suggesting a likely root cause or a useful config setting. The validator still
owns the actual outcome.


## Inspect Filetree Risks

MeerQat also checks for broader filetree problems inspired by the same class
of checks used in `nViz`: empty directories, extension inventory, and
confusingly similar directory names.


In [12]:
filetree_root = pathlib.Path("../..") / "tests" / "data" / "filetree_risks"
filetree_report = validate_dataset(filetree_root / "dataset")

{
    "status": filetree_report.summary.status,
    "filetree_summary": filetree_report.dataset.filetree_summary.to_dict(),
    "filetree_issues": [
        issue.to_dict()
        for issue in filetree_report.issues
        if issue.code.startswith("dataset.")
    ],
}

{'status': 'warn',
 'filetree_summary': {'file_extensions': {'.tiff': 1, '.xml': 1},
  'empty_directories': ['/Users/buntend/Documents/work/meerqat/tests/data/filetree_risks/dataset/empty_dir',
   '/Users/buntend/Documents/work/meerqat/tests/data/filetree_risks/dataset/segment_A',
   '/Users/buntend/Documents/work/meerqat/tests/data/filetree_risks/dataset/segment_B'],
  'similarly_named_directories': [['/Users/buntend/Documents/work/meerqat/tests/data/filetree_risks/dataset/segment_A',
    '/Users/buntend/Documents/work/meerqat/tests/data/filetree_risks/dataset/segment_B']]},
 'filetree_issues': [{'code': 'dataset.empty_directory',
   'severity': 'warning',
   'message': "Empty directory detected at '/Users/buntend/Documents/work/meerqat/tests/data/filetree_risks/dataset/empty_dir'.",
   'plate_id': None,
   'path': '/Users/buntend/Documents/work/meerqat/tests/data/filetree_risks/dataset/empty_dir',
   'remediation': 'Remove the empty directory or confirm it is expected output.'},
  {'

The deterministic issues above come from the filetree scan, and the LLM can
then add higher-level interpretation on top of that structure.


## Customize Validation

If you want to tune the local model or other validation behavior, pass an
explicit `ValidationConfig(...)`.


In [13]:
validation_config = ValidationConfig()
validation_config

ValidationConfig(dataset_id=None, assay_template='phenix_harmony', required_files=(), plate_name_pattern=None, expected_images_per_plate=None, expected_plate_count=None, metadata_plate_column='plate_id', fail_on_warning=False, llm=LLMConfig(enabled=True, provider='instructor', model_alias='tinyllama', repo_id=None, filename=None, cache_dir=None, offline=False, auto_start_server=True, n_ctx=2048, n_threads=8, max_tokens=256, temperature=0.2, base_url='http://127.0.0.1:8000/v1'))

## Use the CLI from a Notebook

In a notebook, a leading `!` runs a shell command. MeerQat installs a
`meerqat` entrypoint, so the CLI stays simple.

`!meerqat validate`
`{pathlib.Path("../..") / "tests" / "data" / "xml_mismatch" / "dataset"}`


In [14]:
!meerqat validate \
  {pathlib.Path("../..") / "tests" / "data" / "xml_mismatch" / "dataset"}

# Meerqat Report: dataset

- Schema version: `1.0.0`
- Generated at: `2026-04-03T20:26:24Z`
- Package version: `0.0.1`
- Status: `fail`
- LLM review: `completed`
- Plates: `1`
- Metadata records: `1`
- Images: `1`

## Filetree

- File extensions:
  - `.tiff`: `1`
  - `.xml`: `1`
- Empty directories: none
- Similarly named directories: none

## Issues

- `error` `plate.xml_folder_mismatch` (Plate_B02): Folder 'Plate_B02' does not match XML plate identifier 'Plate_X99'.

## LLM Findings

_Note: LLM-derived hints and findings can be wrong. Verify them against the underlying dataset before acting on them._

- `medium` `Issue` (Plate_B02): Error: Folder 'Plate_B02' does not match XML plate identifier 'Plate_X99'. The folder 'Plate_B02' does not match the XML plate identifier 'Plate_X99' as it contains a different number of images. This may indicate an issue with the XML file or the metadata record.



## LLM Review

MeerQat runs the same deterministic validation first and then adds LLM review
output. The same happens through the CLI.


If the local model is running, the report includes `llm_hints`,
`llm_findings`, and `llm_review` alongside the deterministic issues. If the
model runtime is unavailable, MeerQat still records that LLM review failure in
the report instead of silently skipping it. LLM summaries can still be wrong,
so verify them against the underlying dataset before acting on them.


## Write Reports

MeerQat can also emit machine-readable and human-readable reports.


In [15]:
with TemporaryDirectory() as tmpdir:
    output_dir = pathlib.Path(tmpdir)
    write_reports(
        broken_report,
        json_path=output_dir / "report.json",
        markdown_path=output_dir / "report.md",
        html_path=output_dir / "report.html",
    )
    generated_files = sorted(path.name for path in output_dir.iterdir())

generated_files

['report.html', 'report.json', 'report.md']